# 1. Setup & Environment
- 공간 시각화 라이브러리(GeoPandas, Matplotlib) 로드
- 한글 폰트(NanumGothic) 및 유니코드 마이너스 깨짐 방지 설정

In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

# 폰트 및 유니코드 환경 설정
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

# 2. Configuration & Cartographic Palette Definitions
- 서울시 집계구 경계 및 핫스팟 데이터 입출력 디렉터리(`output/maps/two_model_by_year`) 설정
- ColorBrewer RdBu 기반 표준 학술 색상 체계 및 텍스트 스타일 정의

In [2]:
# 경로 설정
BASE_DIR = Path("/mnt/cowork/EV")
BOUNDARY_FP = BASE_DIR / "input/raw/집계구_2016/집계구.shp"
OUT_DIR = BASE_DIR / "output/maps/two_model_by_year"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 분석 파라미터 및 모델 라벨
YEARS = [2021, 2022, 2023, 2024]
MODELS = ["2SFCA", "Gravity"]
MODEL_LABEL = {
    "2SFCA": "Gaussian 2SFCA",
    "Gravity": "Gravity Model"
}

# ColorBrewer RdBu 기반 학술 지도 컬러 팔레트
PALETTE = {
    "HOT": "#d73027",       # Hot Spot (Crimson Red)
    "COLD": "#2166ac",      # Cold Spot (Navy Blue)
    "NOT_SIG": "#f0f0f0",   # Not Significant (Light Gray)
    "BORDER": "#bdbdbd",    # 집계구 기본 경계선 색상
    "TEXT": "#222222",      # 헤더 및 라벨 색상
    "MUTED": "#666666"      # 캡션 색상
}

print(f">> 출력 디렉터리: {OUT_DIR}")
print(f">> 연도별 분리 맵 생성 계획: {YEARS}개년 (연도당 1행 2열, 총 {len(YEARS)}장)")

>> 출력 디렉터리: /mnt/cowork/EV/output/maps/two_model_by_year
>> 연도별 분리 맵 생성 계획: [2021, 2022, 2023, 2024]개년 (연도당 1행 2열, 총 4장)


# 3. Boundary & Hotspot Data Loader
- 서울시 14,979개 집계구 폴리곤(EPSG:5179) 로드 및 전처리
- 신규 표준 산출물(`two_model_hotspot_k30_mw.csv`) 우선 탐색 로드

In [3]:
# 1. 집계구 경계 로드
gdf = gpd.read_file(BOUNDARY_FP).set_crs(epsg=5179, allow_override=True)
gdf["TOT_REG_CD"] = gdf["TOT_REG_CD"].astype(str)
gdf_seoul = gdf[gdf["TOT_REG_CD"].str.startswith("11")].copy().reset_index(drop=True)
print(f">> 서울시 집계구 경계 로드 완료: 총 {len(gdf_seoul):,}개 폴리곤")

# 2. 핫스팟 데이터 로드 (_mw 우선 탐색)
fp_hotspot_mw = BASE_DIR / "output/two_model_hotspot_k30_mw.csv"
fp_hotspot_orig = BASE_DIR / "output/two_model_hotspot_k30.csv"

if fp_hotspot_mw.exists():
    fp_hotspot = fp_hotspot_mw
    print(f">> 신규 리팩토링 산출물 로드: {fp_hotspot.name}")
elif fp_hotspot_orig.exists():
    fp_hotspot = fp_hotspot_orig
    print(f">> 기존 원본 산출물 로드: {fp_hotspot.name}")
else:
    raise FileNotFoundError("핫스팟 분석 결과 CSV 파일이 존재하지 않습니다.")

df_hotspot = pd.read_csv(fp_hotspot, dtype={"oa_code": str})

>> 서울시 집계구 경계 로드 완료: 총 19,153개 폴리곤
>> 신규 리팩토링 산출물 로드: two_model_hotspot_k30_mw.csv


# 4. Batch Annual Side-by-Side Map Generation (1 Row x 2 Columns)
- 연도별 독립 피겨(Figure) 생성 (Gaussian 2SFCA vs Gravity Model 좌우 배치)
- 경계선 노이즈 제거를 위한 군집 디졸브(Dissolve) 고속 렌더링
- dissolve 직후 부동소수점 오차로 남는 미세 seam(스펙클) 제거용 소버퍼(+1m/-1m) 스냅 적용
- 하단 통합 범례 배치 및 고해상도(300 DPI) 이미지(`{year}_mw.png`) 개별 저장

In [4]:
print("=" * 80)
print("RUNNING: ANNUAL SIDE-BY-SIDE CARTOGRAPHIC MAP EXPORT")
print("=" * 80)

# 공통 범례 핸들 정의
legend_handles = [
    mpatches.Patch(color=PALETTE["HOT"], label="Hot Spot (p < 0.05)"),
    mpatches.Patch(color=PALETTE["COLD"], label="Cold Spot (p < 0.05)"),
    mpatches.Patch(facecolor=PALETTE["NOT_SIG"], edgecolor=PALETTE["BORDER"], linewidth=0.5, label="Not Significant"),
]

for year in YEARS:
    fig, axes = plt.subplots(1, len(MODELS), figsize=(14, 7.2), facecolor="white")
    plt.subplots_adjust(top=0.86, bottom=0.12, left=0.04, right=0.96, wspace=0.04)

    for ax, model in zip(axes, MODELS):
        # 해당 모형-연도 데이터 필터링
        sub_df = df_hotspot[(df_hotspot["model"] == model) & (df_hotspot["year"] == year)].set_index("oa_code")
        g = gdf_seoul.copy()
        g["gi_class"] = g["TOT_REG_CD"].map(sub_df["gi_class"]).fillna("Not Sig")

        # 기본 배경 집계구 렌더링 (Not Significant)
        g.plot(ax=ax, color=PALETTE["NOT_SIG"], edgecolor=PALETTE["BORDER"], linewidth=0.15)

        # 유의미한 군집 디졸브 렌더링
        dissolved = g.dissolve(by="gi_class")
        # 인접 집계구 경계가 부동소수점 정밀도 차이로 완전히 안 붙어서 dissolve가
        # 매끈한 폴리곤 대신 미세하게 끊긴 조각(스펙클)을 남기는 문제 — 작은 버퍼로 스냅
        dissolved["geometry"] = dissolved.geometry.buffer(1).buffer(-1)
        if "Hot Spot" in dissolved.index:
            dissolved.loc[["Hot Spot"]].plot(ax=ax, color=PALETTE["HOT"], edgecolor="none", alpha=0.9)
        if "Cold Spot" in dissolved.index:
            dissolved.loc[["Cold Spot"]].plot(ax=ax, color=PALETTE["COLD"], edgecolor="none", alpha=0.9)

        ax.set_axis_off()
        ax.set_title(MODEL_LABEL[model], fontsize=13, fontweight="bold", color=PALETTE["TEXT"], pad=8)

    # 연도별 메인 타이틀 및 분석 파라미터 캡션
    fig.suptitle(
        f"{year} Spatial Clustering: Gaussian 2SFCA vs Gravity Model",
        fontsize=15, fontweight="bold", color=PALETTE["TEXT"], y=0.95
    )
    fig.text(
        0.5, 0.90,
        "Weekdays Daytime (11:00-13:00) | Local Getis-Ord Gi* (KNN k=30, p < 0.05)",
        ha="center", fontsize=10, color=PALETTE["MUTED"]
    )

    # 하단 중앙 통합 수평 범례
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=10.5,
        bbox_to_anchor=(0.5, 0.03)
    )

    # 파일 내보내기 (_mw.png)
    out_fp_mw = OUT_DIR / f"{year}_mw.png"
    fig.savefig(out_fp_mw, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [>] {year}년 지도 저장 완료 -> {out_fp_mw.name}")

RUNNING: ANNUAL SIDE-BY-SIDE CARTOGRAPHIC MAP EXPORT
  [>] 2021년 지도 저장 완료 -> 2021_mw.png
  [>] 2022년 지도 저장 완료 -> 2022_mw.png
  [>] 2023년 지도 저장 완료 -> 2023_mw.png
  [>] 2024년 지도 저장 완료 -> 2024_mw.png


# 5. Export Files Verification
- 생성된 4개 연도별 분리 지도 파일의 존재 여부 및 파일 용량(MB) 전수 검증

In [5]:
records = []
for year in YEARS:
    fp = OUT_DIR / f"{year}_mw.png"
    if fp.exists():
        records.append({
            "연도": year,
            "파일명": fp.name,
            "파일 크기(MB)": f"{fp.stat().st_size / (1024 * 1024):.2f}",
            "상태": "정상 생성 완료"
        })
    else:
        records.append({
            "연도": year,
            "파일명": fp.name,
            "파일 크기(MB)": "-",
            "상태": "파일 누락 확인 필요"
        })

df_status = pd.DataFrame(records)
print("=" * 70)
print("             연도별 개별 지도 산출물 내보내기 검증 요약표")
print("=" * 70)
display(df_status)

             연도별 개별 지도 산출물 내보내기 검증 요약표


,연도,파일명,파일 크기(MB),상태
0,2021,2021_mw.png,1.95,정상 생성 완료
1,2022,2022_mw.png,1.97,정상 생성 완료
2,2023,2023_mw.png,1.95,정상 생성 완료
3,2024,2024_mw.png,1.92,정상 생성 완료
